# Rategoan - Training Neural 50M / 100M / 200M di GPU T4 gratis (Colab)

**MEGA-BATCH RAGETAN ROUND 5 FASE 2 + ROUND 6 FASE 2/3 + ROUND 7 FASE 6 + ROUND 8 FASE 1.** Notebook ini melatih checkpoint `raget-neural-massive{50m,100m,200m}.safetensors` di GPU T4 gratis Colab, memakai korpus GABUNGAN jilid 1 (Wikipedia, ~236,4 juta token) + jilid 2 (berita + web, ~421,3 juta token) = **~657,7 juta token total** (lihat `korpus-jilid2-manifest.json` untuk rincian - target 1 miliar belum tercapai karena Release baru memuat 3/140 shard sumber, tinggal tambah shard lewat pipeline FASE 3 Round 8).

## 3 LANGKAH MANUAL (yang lain otomatis)

1. **Runtime > Change runtime type > T4 GPU** (lalu Save).
2. Klik ikon kunci (Secrets) di sidebar kiri Colab, tambahkan secret bernama **`GITHUB_TOKEN`** berisi GitHub Personal Access Token (scope `repo`). Kalau tidak diset, sel di bawah akan minta paste token manual sekali.
3. **Runtime > Run all.** Atur `MODEL_QUEUE` di sel training (mis. `['50m','100m','200m']` untuk antrean otomatis, atau `['100m']` untuk satu ukuran saja) dan `MODE_TUNTAS` sebelum Run all kalau mau ubah default.

## MODE TUNTAS + AUTO-RESUME (Round 8)

- Training berhenti kalau SALAH SATU tercapai: 1 epoch penuh korpus gabungan, target perplexity (opsional), atau `BUDGET_MINUTES_PER_MODEL` sesi ini habis.
- **Sesi Colab mati/putus? Klik Run all lagi** - checkpoint terakhir (di GitHub, ter-auto-commit tiap `COMMIT_EVERY_MINUTES`) dimuat balik, training lanjut dari step terakhir, NOL progres hilang.
- `MODEL_QUEUE` diproses berurutan (mis. 50m dulu sampai budget/target, baru 100m, baru 200m) dalam satu Run all - tidak perlu jalankan notebook terpisah per ukuran.

Preprocessing (unduh dump jilid 1 + jilid 2, extract, clean+dedupe, tokenisasi) makan waktu sekitar 30-45 menit CPU Colab sebelum GPU mulai dipakai - satu kali di awal, dipakai bersama untuk semua ukuran di `MODEL_QUEUE`.

In [ ]:
# @title 0. Cek GPU tersedia
import subprocess
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout)
import torch
print('torch:', torch.__version__)
print('CUDA tersedia:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('PERINGATAN: tidak ada GPU terdeteksi - pastikan Runtime > Change runtime type > T4 GPU sudah dipilih (langkah manual 1).')

In [ ]:
# @title 1. Ambil GitHub token (Secret Colab atau input manual)
import os
GITHUB_TOKEN = None
try:
    from google.colab import userdata
    GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
except Exception:
    GITHUB_TOKEN = None
if not GITHUB_TOKEN:
    import getpass
    GITHUB_TOKEN = getpass.getpass('Secret GITHUB_TOKEN tidak ditemukan - paste GitHub PAT (scope repo) di sini: ')
assert GITHUB_TOKEN, 'Token GitHub wajib diisi untuk clone repo private + unduh aset Release + push hasil.'
os.environ['GITHUB_TOKEN'] = GITHUB_TOKEN
print('Token diterima (', len(GITHUB_TOKEN), 'karakter).')

In [ ]:
# @title 2. Clone repo Rategoan
REPO_OWNER = 'maetalizer-png'
REPO_NAME = 'Rategoan'
BRANCH = 'main'  # @param {type:"string"}

clone_url = 'https://' + GITHUB_TOKEN + '@github.com/' + REPO_OWNER + '/' + REPO_NAME + '.git'
!rm -rf /content/repo
!git clone --branch {BRANCH} --single-branch {clone_url} /content/repo
%cd /content/repo
!git config user.email 'colab-gpu-runner@rategoan.local'
!git config user.name 'Rategoan Colab GPU Runner'
!ls raget/raget-data/neural/
!ls raget/raget-tools/train-massive50m-colab-gpu.py

In [ ]:
# @title 3. Unduh dump Wikipedia dari GitHub Release 'Korpus' (autentikasi PAT - repo private)
import requests, os

ASSET_FILENAME = 'idwiki-latest-pages-articles.xml.bz2'
TAG = 'Korpus'

meta_headers = {'Authorization': 'Bearer ' + GITHUB_TOKEN, 'Accept': 'application/vnd.github+json'}
r = requests.get('https://api.github.com/repos/' + REPO_OWNER + '/' + REPO_NAME + '/releases/tags/' + TAG, headers=meta_headers)
r.raise_for_status()
release = r.json()
asset = next(a for a in release['assets'] if a['name'] == ASSET_FILENAME)
asset_id = asset['id']
print('Aset ditemukan: id=', asset_id, 'ukuran=', asset['size'], 'bytes')

dl_headers = {'Authorization': 'Bearer ' + GITHUB_TOKEN, 'Accept': 'application/octet-stream'}
dump_path = '/content/idwiki-latest-pages-articles.xml.bz2'
with requests.get('https://api.github.com/repos/' + REPO_OWNER + '/' + REPO_NAME + '/releases/assets/' + str(asset_id), headers=dl_headers, stream=True, allow_redirects=True) as resp:
    resp.raise_for_status()
    with open(dump_path, 'wb') as f:
        for chunk in resp.iter_content(chunk_size=8 * 1024 * 1024):
            f.write(chunk)
print('Selesai unduh:', os.path.getsize(dump_path), 'bytes (harus mendekati', asset['size'], ')')

In [ ]:
# @title 4. Ekstrak tokenizer (merges+vocab) LANGSUNG dari metadata checkpoint
# Token ID WAJIB identik dengan yang dipakai waktu checkpoint ini dilatih -
# bpe-tokenizer.json terpisah TIDAK dikomit ke git (lihat wikipedia-korpus-manifest.json),
# tapi merges+vocabEntries-nya tersimpan ulang di metadata checkpoint sendiri.
!python3 raget/raget-tools/extract-tokenizer-from-checkpoint.py raget/raget-data/neural/raget-neural-massive50m.safetensors /content/bpe-tokenizer.json
!ls -la /content/bpe-tokenizer.json

In [ ]:
# @title 5. Extract + bersihkan wikitext (regenerasi korpus - deterministik, sama seperti sandbox)
STOP_AT_WORDS = 150000000  # @param {type:"integer"}
!python3 raget/raget-tools/extract-clean-wikipedia.py /content/idwiki-latest-pages-articles.xml.bz2 /content/idwiki-clean.jsonl {STOP_AT_WORDS}

In [ ]:
# @title 6. Tokenisasi + chunking (window 512) memakai tokenizer checkpoint
!python3 raget/raget-tools/tokenize-chunk-corpus.py /content/idwiki-clean.jsonl /content/bpe-tokenizer.json /content/idwiki-chunks.txt 512

In [ ]:
# @title 7. Rechunk ke window 126 (sama seperti round 3/4/5 CPU - attention lebih murah)
!python3 raget/raget-tools/rechunk-corpus-window.py /content/idwiki-chunks.txt /content/idwiki-chunks-126.txt 126

In [ ]:
# @title 8. Unduh 4 aset korpus JILID 2 dari GitHub Release 'Corpus'
JILID2_TAG = 'Corpus'
JILID2_FILES = ['newspapers-json.tgz', 'train-00000-of-00140.parquet', 'train-00001-of-00140.parquet', 'train-00002-of-00140.parquet']

r2 = requests.get('https://api.github.com/repos/' + REPO_OWNER + '/' + REPO_NAME + '/releases/tags/' + JILID2_TAG, headers=meta_headers)
r2.raise_for_status()
release2 = r2.json()
assets_by_name = {a['name']: a for a in release2['assets']}

os.makedirs('/content/jilid2', exist_ok=True)
for fname in JILID2_FILES:
    asset = assets_by_name[fname]
    out_path = '/content/jilid2/' + fname
    print('Unduh', fname, '(', asset['size'], 'bytes)...')
    with requests.get('https://api.github.com/repos/' + REPO_OWNER + '/' + REPO_NAME + '/releases/assets/' + str(asset['id']), headers=dl_headers, stream=True, allow_redirects=True) as resp:
        resp.raise_for_status()
        with open(out_path, 'wb') as f:
            for chunk in resp.iter_content(chunk_size=8 * 1024 * 1024):
                f.write(chunk)
    actual = os.path.getsize(out_path)
    print('  selesai:', actual, 'bytes (harus =', asset['size'], ') ->', 'OK' if actual == asset['size'] else 'UKURAN TIDAK COCOK!')

In [ ]:
# @title 9. Parse jilid 2 (tgz + 3 parquet) jadi JSONL mentah
!python3 raget/raget-tools/parse-korpus-jilid2.py /content/jilid2/newspapers-json.tgz /content/jilid2/train-00000-of-00140.parquet /content/jilid2/train-00001-of-00140.parquet /content/jilid2/train-00002-of-00140.parquet /content/jilid2

In [ ]:
# @title 10. Bersihkan + dedupe jilid 2 (exact + near MinHash)
!python3 raget/raget-tools/clean-dedupe-korpus-jilid2.py /content/jilid2 /content/jilid2/korpus-jilid2-clean.jsonl /content/jilid2/fase3-clean-report.json

In [ ]:
# @title 11. Tokenisasi jilid 2 + rechunk window 126 + gabung dengan jilid 1
!python3 raget/raget-tools/tokenize-chunk-corpus.py /content/jilid2/korpus-jilid2-clean.jsonl /content/bpe-tokenizer.json /content/jilid2/jilid2-chunks-512.txt 512
!python3 raget/raget-tools/rechunk-corpus-window.py /content/jilid2/jilid2-chunks-512.txt /content/jilid2/jilid2-chunks-126.txt 126
!cat /content/idwiki-chunks-126.txt /content/jilid2/jilid2-chunks-126.txt > /content/korpus-gabungan-126.txt
!wc -l /content/korpus-gabungan-126.txt

In [ ]:
# @title 12. TRAINING GPU - MODE TUNTAS, antrean 50m/100m/200m, auto-resume+auto-commit
MODEL_QUEUE = ['50m', '100m']  # @param {type:"raw"}
BUDGET_MINUTES_PER_MODEL = 300  # @param {type:"number"}
BATCH_SIZE = 96  # @param {type:"number"}
TARGET_PERPLEXITY = 0  # @param {type:"number"}
COMMIT_EVERY_MINUTES = 15  # @param {type:"number"}
PUSH_BRANCH = 'colab-gpu-training'  # @param {type:"string"}
CORPUS_FILE = '/content/korpus-gabungan-126.txt'  # gabungan jilid 1 + jilid 2

for model_size in MODEL_QUEUE:
    print('\n\n========== TRAINING', model_size, '==========')
    !python3 raget/raget-tools/train-massive-colab-gpu.py {model_size} {BUDGET_MINUTES_PER_MODEL} {BATCH_SIZE} {CORPUS_FILE} {TARGET_PERPLEXITY} {COMMIT_EVERY_MINUTES} {PUSH_BRANCH}

print('\nAntrean selesai:', MODEL_QUEUE)
print('Sesi mati di tengah jalan? Klik Run all lagi - checkpoint terakhir (auto-commit tiap', COMMIT_EVERY_MINUTES, 'menit) dimuat balik otomatis, training lanjut, nol progres hilang.')

In [ ]:
# @title 13. Commit + push final (jaring pengaman - skrip training sudah auto-commit tiap interval)
for model_size in MODEL_QUEUE:
    ckpt = 'raget/raget-data/neural/raget-neural-massive{}.safetensors'.format(model_size)
    report = 'raget/raget-devlog/neural/training-report-massive{}-round8-colab-gpu.json'.format(model_size)
    !git add {ckpt} {report}
!git commit -m "Round 8: sesi Colab selesai - queue {MODEL_QUEUE}"
!git push origin HEAD:{PUSH_BRANCH}
print('Selesai. Semua checkpoint di MODEL_QUEUE sudah di-push ke branch:', PUSH_BRANCH)
print('Merge branch ini ke main lewat Pull Request di GitHub, atau minta sesi Rategoan berikutnya untuk merge otomatis.')

## Catatan jujur

- Sel 5-7 dan 9-11 meregenerasi korpus jilid 1 (Wikipedia) dan jilid 2 (berita+web) dari Release GitHub secara DETERMINISTIK (skrip identik dengan yang dipakai di sandbox), bukan mengunduh korpus siap-pakai - satu-satunya cara membawa data besar ke Colab tanpa upload manual berulang.
- Total korpus gabungan saat ini (jilid 1 + jilid 2) TERUKUR ~657,7 juta token - lihat `raget-data/jsonl/external/korpus-jilid2-manifest.json` untuk rincian per sumber. BELUM 1 miliar (Release jilid 2 baru memuat 3 dari 140 shard sumber) - menambah jilid 3 dst tinggal jalankan pipeline `raget-tools/parse-korpus-jilid2.py` + `clean-dedupe-korpus-jilid2.py` dengan tag Release baru (lihat `docs/tambah-data-baru.md`).
- Token/detik GPU T4 sungguhan HANYA diketahui setelah sel 12 selesai jalan - lihat `training-report-massive{50m,100m,200m}-round8-colab-gpu.json` untuk angka nyata, bukan estimasi notebook ini.
- fp16 (bukan bf16) dipakai karena GPU T4 = arsitektur Turing, tidak punya tensor core bf16 yang baik seperti Ampere+.
- MODE TUNTAS berhenti di epoch penuh ATAU target perplexity ATAU budget habis - budget habis BUKAN berarti training selesai, tinggal Run all lagi untuk lanjut (checkpoint+step diakumulasi otomatis dari metadata checkpoint sendiri).
- Checkpoint hasil Colab di-push ke branch terpisah (`colab-gpu-training` secara default) supaya tidak bentrok dengan commit yang sedang berjalan di sandbox lain - perlu di-merge manual (PR) atau oleh sesi Rategoan berikutnya.